In [12]:
from pylib.setup import *
setup_notebook()

from assumptions import smr_tech, SMR_CF, solar_tech, wind_tech, battery
from model import DatacenterDemand, GridSupply, KKSupply, VESupply

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Scenarieanalyse: KK vs VE — 2022–2025, x ∈ {25 %, 50 %, 75 %}

*`1. konfiguration`*

In [ ]:
# 1. konfiguration
YEARS         = [2022, 2023, 2024, 2025]
X_VALUES      = [0.25, 0.50, 0.75]
STORAGE_HOURS = 4

# Installed capacity per year — from 1_input.ipynb VE() output
# "Index denominator is: SolarPowerCapacity / OnshoreWindCapacity: <value> (last observed)"
SOLAR_CAP_MW = {2022: 2_752.88, 2023: 3_528.93, 2024: 3_913.24, 2025: 4_955.54}
WIND_CAP_MW  = {2022: 4_715.28, 2023: 4_801.21, 2024: 4_861.83, 2025: 4_878.48}

*`2. model runs`*

In [14]:
# 2. model runs  (12 LP solves — ~seconds total)
pat  = pathlib.Path('variation_patterns')
rows = []

for year in YEARS:
    solar_cf = np.loadtxt(pat / f'PV_VE_{year}_{year+1}.txt') / SOLAR_CAP_MW[year]
    wind_cf  = np.loadtxt(pat / f'WL_VE_{year}_{year+1}.txt') / WIND_CAP_MW[year]
    prices   = np.loadtxt(pat / f'wp_{year}_{year+1}.txt')    / 7.46

    print(f'{year}  priser: mean={prices.mean():.1f}  min={prices.min():.1f}  max={prices.max():.1f} EUR/MWh')

    for x in X_VALUES:
        demand = DatacenterDemand(demand_mw=1_000.0, x=x)
        grid   = GridSupply(prices, demand)
        kk     = KKSupply(smr_tech, demand, capacity_factor=SMR_CF)
        ve     = VESupply(solar_cf, wind_cf, solar_tech, wind_tech, battery,
                          demand, prices=prices, storage_hours=STORAGE_HOURS)

        r_kk = kk.result(grid)
        r_ve = ve.result(grid)

        rows.append({
            'år':             year,
            'x':              x,
            'kk_lcoe':        r_kk.lcoe,
            've_lcoe':        r_ve.lcoe,
            'gap_lcoe':       r_ve.lcoe - r_kk.lcoe,
            'kk_total_meur':  r_kk.annual_total_cost / 1e6,
            've_total_meur':  r_ve.annual_total_cost / 1e6,
            'gap_meur':      (r_ve.annual_total_cost - r_kk.annual_total_cost) / 1e6,
            've_export_meur': r_ve.annual_export_revenue / 1e6,
            've_grid_meur':   r_ve.annual_grid_cost / 1e6,
        })
        print(f'  x={x:.0%}  KK {r_kk.lcoe:.2f}  VE {r_ve.lcoe:.2f}  gap {r_ve.lcoe - r_kk.lcoe:+.2f} €/MWh')

df = pd.DataFrame(rows)
print('\nFærdig.')

2022  priser: mean=215.1  min=-11.1  max=868.4 EUR/MWh
  x=25%  KK 74.60  VE -3.49  gap -78.09 €/MWh
  x=50%  KK 74.60  VE 29.38  gap -45.22 €/MWh
  x=75%  KK 74.60  VE 78.43  gap +3.83 €/MWh
2023  priser: mean=84.6  min=-299.6  max=524.0 EUR/MWh
  x=25%  KK 74.60  VE 56.30  gap -18.30 €/MWh
  x=50%  KK 74.60  VE 69.34  gap -5.26 €/MWh
  x=75%  KK 74.60  VE 114.62  gap +40.02 €/MWh
2024  priser: mean=70.8  min=-60.0  max=936.0 EUR/MWh
  x=25%  KK 74.60  VE 67.67  gap -6.92 €/MWh
  x=50%  KK 74.60  VE 148.26  gap +73.66 €/MWh
  x=75%  KK 74.60  VE 250.96  gap +176.36 €/MWh
2025  priser: mean=81.6  min=-30.7  max=583.5 EUR/MWh
  x=25%  KK 74.60  VE 60.34  gap -14.25 €/MWh
  x=50%  KK 74.60  VE 85.86  gap +11.27 €/MWh
  x=75%  KK 74.60  VE 137.35  gap +62.76 €/MWh

Færdig.


*`3. resultattabel`*

In [15]:
# 3. resultattabel
tbl = (
    df
    .assign(x=lambda d: d['x'].map(lambda v: f'{v:.0%}'))
    .set_index(['år', 'x'])
    [['kk_total_meur', 've_total_meur', 'gap_meur']]
    .rename(columns={
        'kk_total_meur': 'KK (M€/yr)',
        've_total_meur': 'VE (M€/yr)',
        'gap_meur':      'VE−KK (M€/yr)',
    })
)

tbl.style.format({
    'KK (M€/yr)':     '{:.1f}',
    'VE (M€/yr)':     '{:.1f}',
    'VE−KK (M€/yr)':  '{:+.1f}',
})